In [1]:
import torch
from torch import nn
from torch.optim import Adam
import torchvision
from torchvision.transforms import v2 as tform
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import precision_score, accuracy_score, recall_score, f1_score
import numpy as np

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 512
TRAINING_SEED = 12
NUM_EPOCHS = 10
LR = 1e-3
TRAIN_SPLIT = 0.8
TFORM = tform.Compose([    
    tform.ToImage(),    
    tform.ToDtype(torch.float32, scale=True)
])

In [3]:
import torch
import numpy as np
import random

def set_reproducibility(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)    
    torch.manual_seed(seed)
        
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)        
        
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False        

In [4]:
dataset = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=TFORM)
train_split = int(0.8*len(dataset))
val_split = len(dataset)- train_split

g = torch.Generator()
g.manual_seed(TRAINING_SEED)

training_set, validation_set = random_split(dataset, [train_split, val_split], generator=g)

train_dataloader = DataLoader(training_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, generator=g) 
val_dataloader   = DataLoader(validation_set, batch_size=BATCH_SIZE, shuffle=False)

In [5]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*6*6, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.fc(self.conv(x))

In [6]:
model = SimpleCNN()
model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
opt = Adam(model.parameters(), lr=LR)

In [7]:
from client.traintrack import TrainTrackClient
from client.enums import SplitEnum, MetricEnum
client = TrainTrackClient()
if client.create_model("CIFAR10", "Project_Demo") is None:
    client.get_model("CIFAR10", "Project_Demo")

hyperparameters = {
    'batch size' : BATCH_SIZE,
    'seed' : TRAINING_SEED,
    'epochs' : NUM_EPOCHS,
    'learning rate' : LR,
    'training split' :TRAIN_SPLIT
}
client.create_run(hyperparameters, note="SimpleCNN")

'7d5969dc-2d6a-46a0-bd39-30ffe3428c3a'

In [8]:
def send_data_to_backend(client:TrainTrackClient, step, split:SplitEnum, loss, acc, prec, rec, f1):
    client.log_loss(step, split, loss)
    metrics = [{
        'step' : step,
        'split' : split,
        'metric_name' : MetricEnum.accuracy,
        'value' : acc
    },
    {
        'step' : step,
        'split' : split,
        'metric_name' : MetricEnum.precision,
        'value' : prec
    },
    {
        'step' : step,
        'split' : split,
        'metric_name' : MetricEnum.recall,
        'value' : rec
    },
    {
        'step' : step,
        'split' : split,
        'metric_name' : MetricEnum.f1_score,
        'value' : f1
    }
    ]
    client.log_metrics(metrics)

In [9]:
for ep in range(NUM_EPOCHS):

    model.train()
    train_preds, train_labels = [], []
    train_loss = 0.0

    for imgs, labels in train_dataloader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        outputs = model(imgs)
        loss = criterion(outputs, labels)

        opt.zero_grad()
        loss.backward()
        opt.step()

        train_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)
        train_preds.extend(preds.cpu().numpy())
        train_labels.extend(labels.cpu().numpy())
    
    train_accuracy = np.mean(np.array(train_preds) == np.array(train_labels))
    train_precision = precision_score(train_labels, train_preds, average='macro')
    train_recall = recall_score(train_labels, train_preds, average='macro')
    train_f1 = f1_score(train_labels, train_preds, average='macro')
    send_data_to_backend(client, ep, SplitEnum.train,  train_loss/len(train_dataloader), train_accuracy, train_precision, train_recall, train_f1)
    # VALIDATION LOOP

    model.eval()
    val_preds, val_labels = [], []
    val_loss = 0.0

    for imgs, labels in val_dataloader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        outputs = model(imgs)
        loss = criterion(outputs, labels)

        val_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)
        val_preds.extend(preds.cpu().numpy())
        val_labels.extend(labels.cpu().numpy())
    
    val_accuracy = np.mean(np.array(val_preds) == np.array(val_labels))
    val_precision = precision_score(val_labels, val_preds, average='macro')
    val_recall = recall_score(val_labels, val_preds, average='macro')
    val_f1 = f1_score(val_labels, val_preds, average='macro')
    send_data_to_backend(client, ep, SplitEnum.validation,  val_loss/len(val_dataloader), val_accuracy, val_precision, val_recall, val_f1)

    print(f"\n📊 Epoch [{ep+1}/{NUM_EPOCHS}]")
    print(f"TRAIN  Loss: {train_loss/len(train_dataloader):.4f}")
    print(f"       Acc: {train_accuracy:.4f} | Prec: {train_precision:.4f} | Rec: {train_recall:.4f} | F1: {train_f1:.4f}")
    print(f"VAL    Acc: {val_accuracy:.4f} | Prec: {val_precision:.4f} | Rec: {val_recall:.4f} | F1: {val_f1:.4f}")

client.complete_run()


📊 Epoch [1/10]
TRAIN  Loss: 1.8638
       Acc: 0.3265 | Prec: 0.3211 | Rec: 0.3266 | F1: 0.3131
VAL    Acc: 0.3962 | Prec: 0.4269 | Rec: 0.3954 | F1: 0.3815

📊 Epoch [2/10]
TRAIN  Loss: 1.5349
       Acc: 0.4510 | Prec: 0.4463 | Rec: 0.4510 | F1: 0.4469
VAL    Acc: 0.4783 | Prec: 0.4839 | Rec: 0.4790 | F1: 0.4698

📊 Epoch [3/10]
TRAIN  Loss: 1.3988
       Acc: 0.5015 | Prec: 0.4968 | Rec: 0.5015 | F1: 0.4980
VAL    Acc: 0.5102 | Prec: 0.5182 | Rec: 0.5108 | F1: 0.4952

📊 Epoch [4/10]
TRAIN  Loss: 1.3127
       Acc: 0.5336 | Prec: 0.5284 | Rec: 0.5336 | F1: 0.5300
VAL    Acc: 0.5312 | Prec: 0.5301 | Rec: 0.5306 | F1: 0.5178

📊 Epoch [5/10]
TRAIN  Loss: 1.2667
       Acc: 0.5519 | Prec: 0.5468 | Rec: 0.5519 | F1: 0.5485
VAL    Acc: 0.5568 | Prec: 0.5681 | Rec: 0.5563 | F1: 0.5580

📊 Epoch [6/10]
TRAIN  Loss: 1.2094
       Acc: 0.5726 | Prec: 0.5680 | Rec: 0.5726 | F1: 0.5697
VAL    Acc: 0.5522 | Prec: 0.5882 | Rec: 0.5529 | F1: 0.5470

📊 Epoch [7/10]
TRAIN  Loss: 1.1639
       Acc: 0.59

{'rows_updated': 1}